In [1]:
from genolearn import DataLoader
from genolearn.models.classification import RandomForestClassifier
from genolearn.logger import msg

import numpy as np

dataloader = DataLoader('data-low-memory', 'raw-data/meta-data.csv', 'Accession', 'Region', 'Year')

In [2]:
fisher = dataloader.load_feature_selection('fisher-score.npz')
orders = fisher.rank()

K           = [100, 1000, 10000, 100000, 1000000]
max_depths  = range(5, 51, 5)

predictions = []

for year in reversed(range(2014, 2019)):
    features         = orders[str(year)][:max(K)]
    X_train, Y_train, X_test, Y_test = dataloader.load_train_test(range(year, 2019), [2019], features = features)
    
    predictions_k = []

    for k in K:

        predictions_depth = []

        for max_depth in max_depths:
            
            predictions_seed = []

            for seed in range(10):
                model = RandomForestClassifier(max_depth = max_depth, random_state = seed, class_weight = 'balanced')
                model.fit(X_train, Y_train)
                predictions_seed.append(model.predict(X_test))

            predictions_depth.append(predictions_seed)

        predictions_k.append(predictions_depth)

    predictions.append(predictions_k)

np.savez_compressed('random-forest.npz', predictions = predictions, K = K, max_depths = max_depths)


[2022-03-23 11:40:50] 2018
[2022-03-23 11:43:02] 2017
[2022-03-23 11:46:29] 2016
[2022-03-23 11:52:05] 2015
[2022-03-23 11:58:08] 2014


In [14]:
np.array(predictions).shape

(5, 2, 10, 10, 415)

In [6]:
features

(1000,)

In [3]:
fisher

{'2014': array([0.50747717, 0.43456239, 0.39609989, ..., 0.76318984, 0.76318984,
        0.02099565]),
 '2015': array([0.45390726, 0.36528655, 0.31315239, ..., 0.74352395, 0.73504786,
        0.07787976]),
 '2016': array([0.13913923, 0.11830337, 0.10573812, ..., 0.35855147, 0.35637611,
        0.02487439]),
 '2017': array([0.07214356, 0.06279111, 0.05697376, ..., 0.13541406, 0.13463194,
        0.01387956]),
 '2018': array([0.01812864, 0.01654081, 0.01534129, ..., 0.02061044, 0.02050633,
        0.01069613]),
 '2019': array([0.00626854, 0.0060885 , 0.00578929, ..., 0.00132218, 0.00133474,
        0.01113221])}

In [5]:
fisher.rank()

{'2014': array([11544449,  8961028,  2080733, ...,  6348269,  6348268,  9598966]),
 '2015': array([1829634,  796169, 9247837, ..., 9079733, 3623567, 3411182]),
 '2016': array([ 1485036, 11664233, 11681794, ..., 10152983,  6050098,  7584755]),
 '2017': array([ 2092652,  3671550,  1081146, ..., 10548114, 10964113,   199229]),
 '2018': array([ 2000198, 11722485,  9774307, ...,  1091621,   554752,  5129403]),
 '2019': array([ 7655096,  4307290,  7641598, ..., 10359719,  8433807, 10265938])}